# ChestX-ray14 모델 학습 (Kaggle)

DenseNet-121 gamma 0/1/2 × 5-Fold → 최적 gamma로 EfficientNet-B0 × 5-Fold 를 GPU 2개에 자동 배정해 학습합니다.

**처음 한 번 설정 (오른쪽 패널)**
1. Session options → Accelerator: **GPU T4 x2**, Internet: **On** (ImageNet 사전학습 가중치 다운로드에 필요)
2. Add Input → Datasets: **NIH Chest X ray 14 (224x224 resized)** (khanfashee)
3. Add Input → Your Datasets: **chest-xray14-code** (`chest-xray14-code.zip`을 비공개 Dataset으로 올린 것)
   - 코드(`src/`, `configs/`)를 고치면 zip을 다시 만들어 데이터셋 **New Version**으로 올려야 반영됨

**실행 방법**
- 처음: `RUN_SMOKE_TEST = True` → 위에서부터 셀을 하나씩 실행 (Shift + Enter) → `4) 동작 확인` 셀에서 예상 학습 시간 확인
- 본 학습: `RUN_SMOKE_TEST = False` → 오른쪽 위 **Save Version → Save & Run All (Commit)** → **Stop session** → 창 닫아도 됨
- 이어서: Add Input → Your Work → 이 노트북의 **가장 최근 버전 Output** 추가 → 다시 Save & Run All
  (끝난 Fold는 건너뛰고, 중단된 Fold는 이어서 학습. `5) 본 학습` 셀에 `전체 완료`가 나올 때까지 반복)

In [ ]:
# ===== 설정 =====
RUN_SMOKE_TEST = True       # True: 동작 확인 + 속도 측정, False: 본 학습
TIME_LIMIT_HOURS = 11       # Kaggle 세션 12시간 중 준비·저장 시간을 뺀 학습 시간
SMOKE_LIMIT = 10000         # 동작 확인 때 쓸 학습 이미지 수 (검증은 1/4)

In [ ]:
# 1) 코드 가져오기 — 코드 데이터셋(chest-xray14-code)을 /tmp/repo로 복사
#    (Kaggle Input은 읽기 전용이라 쓰기 가능한 /tmp로 복사)
import glob, os, shutil

found = glob.glob("/kaggle/input/**/src/training/run_queue.py", recursive=True)
print(found)
assert found, "코드 데이터셋을 찾지 못했습니다 (Add Input → Your Datasets → chest-xray14-code 추가 확인)"
assert len(found) == 1, "코드 데이터셋이 여러 개 연결되어 있습니다 (하나만 남기기)"

code_root = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
shutil.rmtree("/tmp/repo", ignore_errors=True)
shutil.copytree(code_root, "/tmp/repo")

%cd /tmp/repo
!ls
!pip install -q pyyaml
!nvidia-smi -L              # Tesla T4 두 줄이 나와야 함

In [ ]:
# 2) 이미지를 Kaggle 서버 디스크(/tmp)로 복사 (내 컴퓨터로 받는 것 아님) — /kaggle/input은 파일마다 읽기 지연이 커서
#    학습 중 GPU가 이미지를 기다림. 처음에 병렬로 한 번 복사해 두면 학습 중 읽기가 빨라짐
#    (메타데이터의 Image Path = images/images-224/<파일명> 구조에 맞춰 복사)
import time
from concurrent.futures import ThreadPoolExecutor

hits = glob.glob("/kaggle/input/**/images-224/00000001_000.png", recursive=True)
assert hits, "00000001_000.png를 찾지 못했습니다 (NIH 데이터셋 추가 여부 확인)"
src_dir = os.path.dirname(hits[0])

RAW = "/tmp/raw"
dst_dir = f"{RAW}/images/images-224"
if os.path.islink(f"{RAW}/images"):                   # 예전 방식(링크)이 남아 있으면 제거
    os.remove(f"{RAW}/images")
os.makedirs(dst_dir, exist_ok=True)

names = [n for n in os.listdir(src_dir) if n.endswith(".png")]
todo = [n for n in names if not os.path.exists(f"{dst_dir}/{n}")]
print(f"이미지 {len(names):,}장 중 {len(todo):,}장 복사 시작")

def copy(name):
    tmp = f"{dst_dir}/{name}.part"                     # 중간에 끊겨도 깨진 파일이 남지 않게
    shutil.copyfile(f"{src_dir}/{name}", tmp)
    os.replace(tmp, f"{dst_dir}/{name}")

t0 = time.time()
with ThreadPoolExecutor(64) as ex:
    for i, _ in enumerate(ex.map(copy, todo), 1):
        if i % 10000 == 0:
            print(f"  {i:,}장 | {i / (time.time() - t0):,.0f}장/s")
print(f"복사 완료: {time.time() - t0:.0f}s")

n_local = len([n for n in os.listdir(dst_dir) if n.endswith(".png")])
print("로컬 이미지 수:", n_local)                     # 112120이어야 함
assert n_local == len(names), "복사된 이미지 수가 다릅니다"

OUT = "/kaggle/working/outputs"
REPORTS = "/kaggle/working/reports"
SETS = [f"raw_dir={RAW}", f"output_dir={OUT}", "num_workers=2"]
print("RAW =", RAW)

In [ ]:
# 3) 이전 버전 결과 이어받기 (이전 버전 Output을 Input으로 추가했을 때만)
os.makedirs(OUT, exist_ok=True)
previous = [d for d in glob.glob("/kaggle/input/**/outputs", recursive=True)
            if glob.glob(f"{d}/*/fold_*")]
print("이어받을 결과:", previous or "없음 (처음 실행)")
for d in previous:
    shutil.copytree(d, OUT, dirs_exist_ok=True)
!ls {OUT}

In [ ]:
# 4) 동작 확인 + 속도 측정 (RUN_SMOKE_TEST = True일 때만, 몇 분)
#    결과는 outputs/_smoke에 따로 저장되고 본 학습·요약에는 포함되지 않음
import pandas as pd

if RUN_SMOKE_TEST:
    shutil.rmtree(f"{OUT}/_smoke", ignore_errors=True)   # 이전 동작 확인 결과가 있으면 건너뛰므로 삭제
    args = " ".join(f"--set {s}" for s in SETS)
    !python -m src.training.run --folds 0 --limit {SMOKE_LIMIT} --set epochs=1 {args}

    # 예상 학습 시간 (준비 시간이 포함돼 실제보다 약간 길게 나옴)
    hist = glob.glob(f"{OUT}/_smoke/*/fold_0/history.csv")
    if hist:
        sec = pd.read_csv(hist[0])["seconds"].iloc[0]
        n_smoke = SMOKE_LIMIT + SMOKE_LIMIT // 4              # 학습 + 검증
        epoch_min = sec * 95394 / n_smoke / 60                # Fold 하나 = 학습 76,315 + 검증 19,079
        fold_h = epoch_min * 15 / 60                          # Early stopping으로 보통 12~20 Epoch
        total_h = fold_h * 20 / 2                             # 작업 20개 ÷ GPU 2개
        print(f"\n측정: {n_smoke:,}장에 {sec:.0f}s ({n_smoke / sec:.0f}장/s)")
        print(f"예상: 1 Epoch ≈ {epoch_min:.0f}분 | Fold 1개 ≈ {fold_h:.1f}시간 | 전체 ≈ {total_h:.0f}시간 (Kaggle 주간 GPU 약 30시간)")

In [ ]:
# 5) 본 학습 — GPU 2개에 작업 자동 배정, TIME_LIMIT_HOURS가 되면 안전하게 멈춤
import subprocess

if not RUN_SMOKE_TEST:
    cmd = ["python", "-m", "src.training.run_queue", "--gpus", "0", "1",
           "--time-limit-hours", str(TIME_LIMIT_HOURS), "--report-dir", REPORTS]
    for s in SETS:
        cmd += ["--set", s]
    code = subprocess.run(cmd).returncode
    print({0: "전체 완료", 75: "시간 제한으로 멈춤 → 이 버전 Output을 Input으로 추가하고 다시 실행",
           1: "실패한 작업 있음 → outputs/_logs 확인"}.get(code, f"종료 코드 {code}"))

In [ ]:
# 6) 결과 확인
from pathlib import Path
from IPython.display import Markdown, display

md_path = Path(REPORTS) / "gamma_comparison.md"
if md_path.exists():
    display(Markdown(md_path.read_text()))
    print((Path(REPORTS) / "selected_gamma.json").read_text())
!ls {OUT}/_logs 2>/dev/null | tail -5